<h1>Merging data</h1>

<p>There are two ways to combine datasets in GeoPandas – attribute joins and spatial joins.</p>

<p>In an attribute join, a <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.html" title="geopandas.GeoSeries"><code>GeoSeries</code></a> or <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a> is combined with a regular
<a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.html#pandas.Series" title="(in pandas v2.3.0)"><code>pandas.Series</code></a> or
<a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html" title="(in pandas v2.3.0)"><code>pandas.DataFrame</code></a> based on a common variable. This is analogous to normal merging or joining in <em>pandas</em>.</p>

<p>In a spatial join, observations from two <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.html" title="geopandas.GeoSeries"><code>GeoSeries</code></a> or <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a> are combined based on their spatial relationship to one another.</p>

<p>In the following examples, these datasets are used:</p>

In [13]:
import geopandas
import geodatasets

In [14]:
chicago = geopandas.read_file(geodatasets.get_path("geoda.chicago_commpop"))
groceries = geopandas.read_file(geodatasets.get_path("geoda.groceries"))

In [15]:
# For attribute join
chicago_shapes = chicago[['geometry', 'NID']]
chicago_names = chicago[['community', 'NID']]

In [16]:
# For spatial join
chicago = chicago[['geometry', 'community']].to_crs(groceries.crs)

# <h2>Appending</h2>

<p>Appending <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a> and <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.html" title="geopandas.GeoSeries"><code>GeoSeries</code></a> uses pandas <a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.concat.html" title="(in pandas v2.3.0)"><code>concat()</code></a> function. Keep in mind, that appended geometry columns needs to have the same CRS.</p>

In [17]:
import pandas as pd

In [18]:
# Appending GeoSeries
joined = pd.concat([chicago.geometry, groceries.geometry])

In [19]:
# Appending GeoDataFrames
douglas = chicago[chicago.community == 'DOUGLAS']
oakland = chicago[chicago.community == 'OAKLAND']
douglas_oakland = pd.concat([douglas, oakland])

# <h2>Attribute joins</h2>

<p>Attribute joins are accomplished using the <a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.merge.html" title="(in pandas v2.3.0)"><code>merge()</code></a> method.
In general, it is recommended to use the <code>merge()</code> method called from the spatial dataset. With that said, the stand-alone
<a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.merge.html" title="(in pandas v2.3.0)"><code>pandas.merge()</code></a> function will work if the <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a> is in the <code>left</code> argument; if a
<a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html" title="(in pandas v2.3.0)"><code>DataFrame</code></a> is in the <code>left</code> argument and a <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a> is in the <code>right</code> position, the result will no longer be a
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a>.</p>

<p>For example, consider the following merge that adds full names to a
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a> that initially has only area ID for each geometry by merging it with a
<a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html#pandas.DataFrame" title="(in pandas v2.3.0)"><code>DataFrame</code></a>.</p>

In [20]:
# `chicago_shapes` is GeoDataFrame with community shapes and area IDs
chicago_shapes.head()

,geometry,NID
0,"MULTIPOLYGON (((-87.60914 41.84469, -87.60915 ...",35
1,"MULTIPOLYGON (((-87.59215 41.81693, -87.59231 ...",36
2,"MULTIPOLYGON (((-87.62880 41.80189, -87.62879 ...",37
3,"MULTIPOLYGON (((-87.60671 41.81681, -87.60670 ...",38
4,"MULTIPOLYGON (((-87.59215 41.81693, -87.59215 ...",39


In [21]:
# chicago_names is DataFrame with community names and area ID
chicago_names.head()

,community,NID
0,DOUGLAS,35
1,OAKLAND,36
2,FULLER PARK,37
3,GRAND BOULEVARD,38
4,KENWOOD,39


In [22]:
# Merge with merge method on shared variable (area ID):
chicago_shapes = chicago_shapes.merge(chicago_names, on='NID')

In [23]:
chicago_shapes.head()

,geometry,NID,community
0,"MULTIPOLYGON (((-87.60914 41.84469, -87.60915 ...",35,DOUGLAS
1,"MULTIPOLYGON (((-87.59215 41.81693, -87.59231 ...",36,OAKLAND
2,"MULTIPOLYGON (((-87.62880 41.80189, -87.62879 ...",37,FULLER PARK
3,"MULTIPOLYGON (((-87.60671 41.81681, -87.60670 ...",38,GRAND BOULEVARD
4,"MULTIPOLYGON (((-87.59215 41.81693, -87.59215 ...",39,KENWOOD


# <h2>Spatial joins</h2>

<p>In a spatial join, two geometry objects are merged based on their spatial relationship to one another.</p>

In [24]:
# One GeoDataFrame of communities, one of grocery stores.
# Want to merge to get each grocery's community.
chicago.head()

,geometry,community
0,"MULTIPOLYGON (((1181573.250 1886828.039, 11815...",DOUGLAS
1,"MULTIPOLYGON (((1186289.356 1876750.733, 11862...",OAKLAND
2,"MULTIPOLYGON (((1176344.998 1871187.546, 11763...",FULLER PARK
3,"MULTIPOLYGON (((1182322.043 1876674.730, 11823...",GRAND BOULEVARD
4,"MULTIPOLYGON (((1186289.356 1876750.733, 11862...",KENWOOD


In [25]:
groceries.head()

,OBJECTID,Ycoord,Xcoord,Status,Address,Chain,Category,geometry
0,16,41.973266,-87.657073,OPEN,"1051 W ARGYLE ST, CHICAGO, IL. 60640",VIET HOA PLAZA,None,MULTIPOINT (1168268.672 1933554.350)
1,18,41.696367,-87.681315,OPEN,"10800 S WESTERN AVE, CHICAGO, IL. 60643-3226",COUNTY FAIR FOODS,None,MULTIPOINT (1162302.618 1832900.224)
2,22,41.868634,-87.638638,OPEN,"1101 S CANAL ST, CHICAGO, IL. 60607-4932",WHOLE FOODS MARKET,None,MULTIPOINT (1173317.042 1895425.426)
3,23,41.877590,-87.654953,OPEN,"1101 W JACKSON BLVD, CHICAGO, IL. 60607-2905",TARGET/SUPER,new,MULTIPOINT (1168996.475 1898801.406)
4,27,41.737696,-87.625795,OPEN,"112 W 87TH ST, CHICAGO, IL. 60620-1318",FOOD 4 LESS,None,MULTIPOINT (1176991.989 1847262.423)


In [26]:
# Execute spatial join
groceries_with_community = groceries.sjoin(chicago, how='inner', predicate='intersects')

In [27]:
groceries_with_community.head()

,OBJECTID,Ycoord,Xcoord,Status,Address,Chain,Category,geometry,index_right,community
0,16,41.973266,-87.657073,OPEN,"1051 W ARGYLE ST, CHICAGO, IL. 60640",VIET HOA PLAZA,None,MULTIPOINT (1168268.672 1933554.350),30,UPTOWN
87,365,41.961707,-87.654058,OPEN,"4355 N SHERIDAN RD, CHICAGO, IL. 60613-1497",JEWEL OSCO,None,MULTIPOINT (1168837.980 1929246.962),30,UPTOWN
90,373,41.963131,-87.656352,OPEN,"4466 N BROADWAY ST, CHICAGO, IL. 60640-5660",TARGET,None,MULTIPOINT (1168471.227 1929825.061),30,UPTOWN
140,582,41.969131,-87.674882,Chicago-Ravenswood,"1800 W Lawrence Ave, Chicago, IL 60640",Mariano's,None,MULTIPOINT (1163502.978 1932264.462),30,UPTOWN
1,18,41.696367,-87.681315,OPEN,"10800 S WESTERN AVE, CHICAGO, IL. 60643-3226",COUNTY FAIR FOODS,None,MULTIPOINT (1162302.618 1832900.224),73,MORGAN PARK


<p>GeoPandas provides two spatial-join functions:</p>

<ul class="simple">
<li><p><a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin.html" title="geopandas.GeoDataFrame.sjoin"><code>GeoDataFrame.sjoin()</code></a>: joins based on binary predicates (intersects, contains, etc.)</p></li>
<li><p><a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin_nearest.html" title="geopandas.GeoDataFrame.sjoin_nearest"><code>GeoDataFrame.sjoin_nearest()</code></a>: joins based on proximity, with the ability to set a maximum search radius.</p></li>
</ul>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>For historical reasons, both methods are also available as top-level functions <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin.html" title="geopandas.sjoin"><code>sjoin()</code></a> and <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin_nearest.html" title="geopandas.sjoin_nearest"><code>sjoin_nearest()</code></a>. It is recommended to use methods as the functions may be deprecated in the future.</p>
</div>

## <h3>Binary predicate joins</h3>

<p>Binary predicate joins are available via <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin.html" title="geopandas.GeoDataFrame.sjoin"><code>GeoDataFrame.sjoin()</code></a>.</p>

<p><a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin.html" title="geopandas.GeoDataFrame.sjoin"><code>GeoDataFrame.sjoin()</code></a> has two core arguments: <code>how</code> and <code>predicate</code>.</p>

<p><strong>predicate</strong></p>

<p>The <code>predicate</code> argument specifies how GeoPandas decides whether or not to join the attributes of one object to another, based on their geometric relationship.</p>

<p>The values for <code>predicate</code> correspond to the names of geometric binary predicates and depend on the spatial index implementation.</p>

<p>The default spatial index in GeoPandas currently supports the following values for <code>predicate</code> which are defined in the
<a href="http://shapely.readthedocs.io/en/latest/manual.html#binary-predicates">Shapely documentation</a>:</p>

<ul class="simple">
<li><p><cite>intersects</cite></p></li>
<li><p><cite>contains</cite></p></li>
<li><p><cite>within</cite></p></li>
<li><p><cite>touches</cite></p></li>
<li><p><cite>crosses</cite></p></li>
<li><p><cite>overlaps</cite></p></li>
</ul>

<p><strong>how</strong></p>

<p>The <cite>how</cite> argument specifies the type of join that will occur and which geometry is retained in the resultant
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a>. It accepts the following options:</p>

<ul class="simple">
<li><p><code>left</code>: use the index from the first (or <cite>left_df</cite>)
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a> that you provide to <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin.html" title="geopandas.GeoDataFrame.sjoin"><code>GeoDataFrame.sjoin()</code></a>; retain only the <cite>left_df</cite> geometry column</p></li>
<li><p><code>right</code>: use index from second (or <cite>right_df</cite>); retain only the <cite>right_df</cite> geometry column</p></li>
<li><p><code>inner</code>: use intersection of index values from both
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a>; retain only the <cite>left_df</cite> geometry column</p></li>
</ul>

<p>Note more complicated spatial relationships can be studied by combining geometric operations with spatial join.
To find all polygons within a given distance of a point, for example, one can first use the
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.buffer.html" title="geopandas.GeoSeries.buffer"><code>buffer()</code></a> method to expand each point into a circle of appropriate radius, then intersect those buffered circles with the polygons in question.</p>

## <h3>Nearest joins</h3>

<p>Proximity-based joins can be done via <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin_nearest.html" title="geopandas.GeoDataFrame.sjoin_nearest"><code>GeoDataFrame.sjoin_nearest()</code></a>.</p>

<p><a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin_nearest.html" title="geopandas.GeoDataFrame.sjoin_nearest"><code>GeoDataFrame.sjoin_nearest()</code></a> shares the <code>how</code> argument with
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin.html" title="geopandas.GeoDataFrame.sjoin"><code>GeoDataFrame.sjoin()</code></a>, and includes two additional arguments: <code>max_distance</code> and <code>distance_col</code>.</p>

<p><strong>max_distance</strong></p>

<p>The <code>max_distance</code> argument specifies a maximum search radius for matching geometries. This can have a considerable performance impact in some cases. If you can, it is highly recommended that you use this parameter.</p>

<p><strong>distance_col</strong></p>
<p>If set, the resultant GeoDataFrame will include a column with this name containing the computed distances between an input geometry and the nearest geometry.</p>